<a href="https://colab.research.google.com/github/Euler912/multi_MDCF/blob/main/Google_Maps_API_TSP_HeldKarp_Ashdod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Maps Routes API → מטריצת מרחקים → TSP מדויק

**מטרה:** לקבל מרחקי כביש בין כתובות, לבנות מטריצה מכוונת, ולמצוא באמצעות Held–Karp את **סדר הביקור האופטימלי ביחס למטריצה שקיבלנו**.

**שלוש אפשרויות:**
- `demo` – נתוני דוגמה סינתטיים; ללא API או חיוב.
- `google_matrix` – קריאת `computeRouteMatrix` יחידה (מרחקי מסלולים ש־Google מחשבת על פי העדפת זמן הנסיעה).
- `google_shorter` – קריאת `computeRoutes` לכל זוג נקודות בכיוון, עם `SHORTER_DISTANCE`, כדי לתת עדיפות למרחק על פני זמן. זו יכולת **ניסיונית** של Google, ואינה אישור מתמטי למסלול הקצר ביותר האפשרי ברשת הכבישים.

**דרישה להפעלת Google:** פרויקט Google Cloud, Billing מופעל, Routes API מופעל ומפתח API שמוגבל לשירות הזה. גוגל מחייבת בהתאם לשימוש. אין להכניס API Key בתאי המחברת או להעלות ל־GitHub.

קישורים: [יצירת API Key](https://developers.google.com/maps/documentation/routes/get-api-key), [Route Matrix](https://developers.google.com/maps/documentation/routes/compute_route_matrix), [Shorter distance](https://developers.google.com/maps/documentation/routes/shorter-distance-routes), [Billing](https://developers.google.com/maps/documentation/routes/usage-and-billing).

In [16]:
import os
import getpass
from functools import lru_cache
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
from IPython.display import display

print("Ready")

Ready


## 1. הגדרות: כתובות ומצב העבודה

אפשר להחליף את כל הכתובות. **הכתובת הראשונה היא המחסן**: נתחיל ממנה ונסיים בה.

בפעם הראשונה השאר `MODE = "demo"` כדי לוודא שהאלגוריתם עובד. כדי למשוך נתונים אמיתיים החלף ל־`google_shorter` או `google_matrix`. **אל תפעיל את שניהם לחינם** — אלו בקשות נפרדות בתשלום לפי שימוש.

In [17]:
ADDRESSES = [
    "האורגים 2, אשדוד, ישראל",      # depot / start + end
    "נחל לכיש 30, אשדוד, ישראל",
    "מורדי הגטאות 10, אשדוד, ישראל",
    "שלמה המלך 3, אשדוד, ישראל",
]

MODE = "demo"  # "demo", "google_matrix", "google_shorter"

assert len(ADDRESSES) >= 2
assert len(set(ADDRESSES)) == len(ADDRESSES)
print("Stops:")
for i, addr in enumerate(ADDRESSES):
    print(f"{i}: {addr}")
print("Mode:", MODE)

Stops:
0: האורגים 2, אשדוד, ישראל
1: נחל לכיש 30, אשדוד, ישראל
2: מורדי הגטאות 10, אשדוד, ישראל
3: שלמה המלך 3, אשדוד, ישראל
Mode: demo


## 2. אימות באמצעות API Key

ב־Colab: בצד שמאל בחר **Secrets (אייקון מפתח)**, הוסף `GOOGLE_MAPS_API_KEY`, הדבק את המפתח והפעל **Notebook access**. אפשר גם להגדיר משתנה סביבה. אם לא, תתבקש להזין את המפתח בצורה מוסתרת.

לקריאת `demo` לא צריך מפתח.

In [18]:
def get_google_key():
    key = os.environ.get("GOOGLE_MAPS_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_MAPS_API_KEY")
        if key:
            return key.strip()
    except Exception:
        pass
    key = getpass.getpass("Google Routes API key (hidden): ").strip()
    if not key:
        raise ValueError("Missing Google Maps API key")
    return key


def google_post(url, payload, key, fields):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": key,
        "X-Goog-FieldMask": fields,
    }
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=45)
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(
            f"Google Routes HTTP {response.status_code}: {response.text[:1200]}"
        ) from exc
    return response.json()


## 3. שני מסלולים אפשריים לקבלת הנתונים

- **`computeRouteMatrix`:** מקבלים את כל הזוגות בבת אחת. Google מחזירה לכל זוג מרחק של המסלול שבחרה לפי העדפת הניתוב (לא בהכרח הקצר ביותר במטרים).
- **`computeRoutes` עם `SHORTER_DISTANCE`:** מבקשים עבור כל זוג מסלול שמעדיף מרחק קצר. התוצאה כוללת גם מסלול ברירת־מחדל; נבחר את זה שמסומן `SHORTER_DISTANCE` כאשר הוא קיים. אם Google לא החזירה מסלול כזה, נדווח על שימוש בברירת המחדל.

מטריצה של 4 נקודות = 16 רכיבי Route Matrix או 12 קריאות נפרדות ב־Shorter Distance (ללא האלכסון). **זה עשוי לעלות כסף.**

In [19]:
def google_distance_matrix(addresses, key):
    n = len(addresses)
    # For n addresses, address-based matrix has 2n address waypoints.
    if n > 25 or n*n > 625:
        raise ValueError("Too many addresses for one Google matrix request")

    body = {
        "origins": [{"waypoint": {"address": a}} for a in addresses],
        "destinations": [{"waypoint": {"address": a}} for a in addresses],
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_UNAWARE",
    }
    items = google_post(
        "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix",
        body, key,
        "originIndex,destinationIndex,status,condition,distanceMeters,duration",
    )
    if not isinstance(items, list):
        raise ValueError(f"Unexpected response: {items}")
    D = np.full((n, n), np.nan)
    T = np.full((n, n), np.nan)
    for e in items:
        i, j = e["originIndex"], e["destinationIndex"]
        if i == j:
            D[i, j] = T[i, j] = 0
            continue
        if e.get("condition") != "ROUTE_EXISTS" or "distanceMeters" not in e:
            raise ValueError(f"No valid route for {i}->{j}: {e}")
        D[i, j] = float(e["distanceMeters"])
        T[i, j] = float(e["duration"].removesuffix("s"))
    np.fill_diagonal(D, 0)
    np.fill_diagonal(T, 0)
    if not (np.isfinite(D).all() and np.isfinite(T).all()):
        raise ValueError("Incomplete route matrix; check the address resolution")
    return D, T


def google_shorter_distance_matrix(addresses, key):
    n = len(addresses)
    D = np.zeros((n, n))
    T = np.zeros((n, n))
    fallback_pairs = []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            body = {
                "origin": {"address": addresses[i]},
                "destination": {"address": addresses[j]},
                "travelMode": "DRIVE",
                "routingPreference": "TRAFFIC_AWARE",
                "requestedReferenceRoutes": ["SHORTER_DISTANCE"],
            }
            data = google_post(
                "https://routes.googleapis.com/directions/v2:computeRoutes",
                body, key,
                "routes.distanceMeters,routes.duration,routes.routeLabels,routes.routeToken",
            )
            routes = data.get("routes", [])
            if not routes:
                raise ValueError(f"No Google route for {i}->{j}: {data}")
            shorter = next(
                (r for r in routes if "SHORTER_DISTANCE" in r.get("routeLabels", [])),
                None,
            )
            if shorter is None:
                fallback_pairs.append((i, j))
                route = next(
                    (r for r in routes if "DEFAULT_ROUTE" in r.get("routeLabels", [])),
                    routes[0],
                )
            else:
                route = shorter
            D[i, j] = float(route["distanceMeters"])
            T[i, j] = float(route["duration"].removesuffix("s"))
    if fallback_pairs:
        print("WARNING: Google did not label a shorter route for pairs:", fallback_pairs)
        print("Default route used for those pairs. Check the map before relying on results.")
    return D, T


## 4. בניית מטריצת המרחקים

הנתונים במצב Demo **מומצאים לצורכי בדיקת הקוד** — הם אינם מרחקים אמיתיים באשדוד. שימוש ב־Google דורש API Key ולקריאות יש עלות אפשרית.

In [20]:
if MODE == "demo":
    if len(ADDRESSES) != 4:
        raise ValueError("Demo mode requires exactly 4 addresses")

    D_m = np.array([
    [   0, 6400, 1300, 4700],
    [5600,    0, 4900, 1600],
    [1100, 4600,    0, 3900],
    [4900, 1700, 3700,    0],
], dtype=float)



    T_s = D_m / 7.0  # fictitious travel times for demonstration only
    print("DEMO ONLY: synthetic distances, not Google Maps data")
elif MODE == "google_matrix":
    api_key = get_google_key()
    D_m, T_s = google_distance_matrix(ADDRESSES, api_key)
elif MODE == "google_shorter":
    api_key = get_google_key()
    D_m, T_s = google_shorter_distance_matrix(ADDRESSES, api_key)
else:
    raise ValueError("MODE must be demo, google_matrix or google_shorter")

print("Distance (km), rows = origin, columns = destination:")
display(pd.DataFrame(D_m/1000, index=ADDRESSES, columns=ADDRESSES).round(3))
print("Duration (min):")
display(pd.DataFrame(T_s/60, index=ADDRESSES, columns=ADDRESSES).round(1))


DEMO ONLY: synthetic distances, not Google Maps data
Distance (km), rows = origin, columns = destination:


,"האורגים 2, אשדוד, ישראל","נחל לכיש 30, אשדוד, ישראל","מורדי הגטאות 10, אשדוד, ישראל","שלמה המלך 3, אשדוד, ישראל"
"האורגים 2, אשדוד, ישראל",0.0,6.4,1.3,4.7
"נחל לכיש 30, אשדוד, ישראל",5.6,0.0,4.9,1.6
"מורדי הגטאות 10, אשדוד, ישראל",1.1,4.6,0.0,3.9
"שלמה המלך 3, אשדוד, ישראל",4.9,1.7,3.7,0.0


Duration (min):


,"האורגים 2, אשדוד, ישראל","נחל לכיש 30, אשדוד, ישראל","מורדי הגטאות 10, אשדוד, ישראל","שלמה המלך 3, אשדוד, ישראל"
"האורגים 2, אשדוד, ישראל",0.0,15.2,3.1,11.2
"נחל לכיש 30, אשדוד, ישראל",13.3,0.0,11.7,3.8
"מורדי הגטאות 10, אשדוד, ישראל",2.6,11.0,0.0,9.3
"שלמה המלך 3, אשדוד, ישראל",11.7,4.0,8.8,0.0


## 5. פתרון TSP מדויק באמצעות Held–Karp

האלגוריתם מוצא את **סדר התחנות האופטימלי לפי מטריצת העלויות שבנינו**, כאשר מתחילים ומסיימים בתחנה 0. הוא עובד גם עם מרחקים **לא סימטריים**. סיבוכיות זמן אקספוננציאלית, לכן כאן מתאים למספר נקודות קטן (למשל 4–15).

In [21]:
def held_karp(D):
    D = np.asarray(D, dtype=float)
    n = D.shape[0]
    if D.ndim != 2 or D.shape != (n, n) or n < 2:
        raise ValueError("D must be a square matrix with n>=2")
    if not np.isfinite(D).all() or (D < 0).any():
        raise ValueError("D must contain non-negative finite distances")

    # Nodes excluding the fixed depot 0: 1, ..., n-1.
    @lru_cache(None)
    def dp(mask, last):
        bit = 1 << (last - 1)
        if not (mask & bit):
            return float("inf"), ()
        if mask == bit:
            return float(D[0, last]), (last,)
        previous_mask = mask ^ bit
        candidates = []
        for prev in range(1, n):
            if previous_mask & (1 << (prev - 1)):
                old_cost, old_path = dp(previous_mask, prev)
                candidates.append((old_cost + D[prev, last], old_path + (last,)))
        return min(candidates, key=lambda item: item[0])

    mask = (1 << (n-1)) - 1
    best_cost = float("inf")
    best_path = None
    for last in range(1, n):
        cost, path = dp(mask, last)
        cost += D[last, 0]
        if cost < best_cost:
            best_cost = float(cost)
            best_path = (0,) + path + (0,)
    assert len(best_path) == n + 1
    return best_path, best_cost

route, total_m = held_karp(D_m)
travel_s = sum(T_s[route[k], route[k+1]] for k in range(len(route)-1))

print("Mode:", MODE)
print("Optimal stop order:", " -> ".join(map(str, route)))
print("Total distance:", round(total_m/1000, 3), "km")
print("Sum of leg duration estimates:", round(travel_s/60, 1), "min")
print("Stops:")
for k, i in enumerate(route):
    print(f"  {k}. [{i}] {ADDRESSES[i]}")


Mode: demo
Optimal stop order: 0 -> 3 -> 1 -> 2 -> 0
Total distance: 12.4 km
Sum of leg duration estimates: 29.5 min
Stops:
  0. [0] האורגים 2, אשדוד, ישראל
  1. [3] שלמה המלך 3, אשדוד, ישראל
  2. [1] נחל לכיש 30, אשדוד, ישראל
  3. [2] מורדי הגטאות 10, אשדוד, ישראל
  4. [0] האורגים 2, אשדוד, ישראל


## 6. פתיחת סדר הביקור ב־Google Maps

הקישור מעביר ל־Google Maps את **סדר התחנות שמצא Held–Karp**. שים לב: Google Maps עשויה לבחור מסלול כבישים שונה, ולכן המרחק שיופיע באפליקציה לא חייב להיות זהה לסכום מטריצת ה־API.

In [22]:
origin = quote(ADDRESSES[route[0]])
destination = quote(ADDRESSES[route[-1]])
waypoints = quote("|".join(ADDRESSES[i] for i in route[1:-1]), safe="")
url = (
    f"https://www.google.com/maps/dir/?api=1&origin={origin}"
    f"&destination={destination}&waypoints={waypoints}&travelmode=driving"
)
print("Google Maps route link:")
print(url)
print("\nCost of ordered legs:")
for i, j in zip(route[:-1], route[1:]):
    print(f"{i} -> {j}: {D_m[i,j]/1000:.3f} km")


Google Maps route link:
https://www.google.com/maps/dir/?api=1&origin=%D7%94%D7%90%D7%95%D7%A8%D7%92%D7%99%D7%9D%202%2C%20%D7%90%D7%A9%D7%93%D7%95%D7%93%2C%20%D7%99%D7%A9%D7%A8%D7%90%D7%9C&destination=%D7%94%D7%90%D7%95%D7%A8%D7%92%D7%99%D7%9D%202%2C%20%D7%90%D7%A9%D7%93%D7%95%D7%93%2C%20%D7%99%D7%A9%D7%A8%D7%90%D7%9C&waypoints=%D7%A9%D7%9C%D7%9E%D7%94%20%D7%94%D7%9E%D7%9C%D7%9A%203%2C%20%D7%90%D7%A9%D7%93%D7%95%D7%93%2C%20%D7%99%D7%A9%D7%A8%D7%90%D7%9C%7C%D7%A0%D7%97%D7%9C%20%D7%9C%D7%9B%D7%99%D7%A9%2030%2C%20%D7%90%D7%A9%D7%93%D7%95%D7%93%2C%20%D7%99%D7%A9%D7%A8%D7%90%D7%9C%7C%D7%9E%D7%95%D7%A8%D7%93%D7%99%20%D7%94%D7%92%D7%98%D7%90%D7%95%D7%AA%2010%2C%20%D7%90%D7%A9%D7%93%D7%95%D7%93%2C%20%D7%99%D7%A9%D7%A8%D7%90%D7%9C&travelmode=driving

Cost of ordered legs:
0 -> 3: 4.700 km
3 -> 1: 1.700 km
1 -> 2: 4.900 km
2 -> 0: 1.100 km


## הערות לאמינות המחקר

1. **Held–Karp מספק אופטימום גלובלי של סדר התחנות ביחס למטריצה הנתונה**; אינו מוכיח כי כל מקטע כביש שגוגל החזירה הוא מסלול כביש קצר גלובלית.
2. Google `SHORTER_DISTANCE` היא תכונה ניסיונית. היא מנסה למזער מרחק נסיעה ויכולה לבחור כבישים לא שגרתיים.
3. הבדלים בין שני הכיוונים לגיטימיים בגלל רחובות חד־סטריים, פניות אסורות וכדומה.
4. הבקשות ל־Google Maps Platform דורשות Billing ועשויות לגרור חיובים. בדוק מכסות/תקציב לפני הרצה על יותר נקודות.
5. ודא שהכתובות מזוהות נכון (בפרט מספר הבית ונקודת הכניסה). אם צריך דיוק ברמת כניסה, העבר `placeId` או קואורדינטות מאומתות במקום מחרוזת כתובת.
6. השלב הבא: להשתמש ב־`D_m` גם עם MDCF ולחשב פער מול `total_m` — על **אותם נתונים בדיוק**.
